<a href="https://colab.research.google.com/github/1BF24CS033/BIS/blob/main/Parallel_Cellular_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import numpy as np
import random


np.random.seed(42)
random.seed(42)

GRID_SIZE = 10
POPULATION_SIZE = 100
INTERSECTIONS = 4

GENERATIONS = 100

MIN_GREEN = 5
MAX_GREEN = 60
YELLOW_TIME = 4

# Each intersection has:
# North-South green duration
# East-West green duration

CHROMOSOME_LENGTH = INTERSECTIONS * 2

SIMULATION_STEPS = 200

arrival_rates = [
    [2.0, 1.5],
    [1.5, 2.0],
    [2.5, 1.0],
    [1.0, 2.5]
]

rng = np.random.default_rng(123)

traffic_data = np.zeros(
    (SIMULATION_STEPS, INTERSECTIONS, 2),
    dtype=int
)

for t in range(SIMULATION_STEPS):

    for i in range(INTERSECTIONS):

        traffic_data[t, i, 0] = rng.poisson(
            arrival_rates[i][0]
        )

        traffic_data[t, i, 1] = rng.poisson(
            arrival_rates[i][1]
        )

def fitness_function(solution):

    total_waiting = 0
    total_remaining = 0

    # Simulate each intersection
    for intersection in range(INTERSECTIONS):

        ns_green = solution[intersection * 2]
        ew_green = solution[intersection * 2 + 1]

        # Complete signal cycle
        cycle_length = (
            ns_green + ew_green + 2 * YELLOW_TIME
        )

        ns_queue = 0
        ew_queue = 0

        for t in range(SIMULATION_STEPS):

            # Generate incoming traffic
            ns_queue += traffic_data[t, intersection, 0]
            ew_queue += traffic_data[t, intersection, 1]

            # Determine current signal phase
            position = t % cycle_length

            # North-South green
            if position < ns_green:

                ns_queue = max(0, ns_queue - 2)

            # North-South yellow
            elif position < ns_green + YELLOW_TIME:
                pass

            # East-West green
            elif position < ns_green + YELLOW_TIME + ew_green:

                ew_queue = max(0, ew_queue - 2)

            # East-West yellow
            else:
                pass

            # Accumulate waiting vehicles
            total_waiting += ns_queue + ew_queue

        total_remaining += ns_queue + ew_queue

    # Minimize waiting time and leftover queues
    fitness = total_waiting + 10 * total_remaining

    return fitness


population = np.random.randint(
    MIN_GREEN,
    MAX_GREEN + 1,
    size=(
        GRID_SIZE,
        GRID_SIZE,
        CHROMOSOME_LENGTH
    )
)

fitness_grid = np.zeros(
    (GRID_SIZE, GRID_SIZE)
)

for i in range(GRID_SIZE):

    for j in range(GRID_SIZE):

        fitness_grid[i, j] = fitness_function(
            population[i, j]
        )

def get_neighbors(grid, x, y):

    neighbors = [
        (x, y),                         # Center
        ((x - 1) % GRID_SIZE, y),       # North
        ((x + 1) % GRID_SIZE, y),       # South
        (x, (y - 1) % GRID_SIZE),       # West
        (x, (y + 1) % GRID_SIZE)        # East
    ]

    return neighbors

def cellular_GA():

    global population, fitness_grid

    for generation in range(GENERATIONS):

        # Synchronous update:
        # All offspring are created from the old population.

        old_population = population.copy()
        old_fitness = fitness_grid.copy()

        new_population = old_population.copy()
        new_fitness = old_fitness.copy()

        for x in range(GRID_SIZE):

            for y in range(GRID_SIZE):

                neighbors = get_neighbors(
                    old_population, x, y
                )

                # Current individual is parent 1
                parent1 = old_population[x, y].copy()

                # Binary tournament selection
                candidates = random.sample(neighbors, 2)

                winner = min(
                    candidates,
                    key=lambda pos: old_fitness[pos]
                )

                parent2 = old_population[
                    winner[0], winner[1]
                ].copy()

                # Uniform crossover
                # Educational substitute for paper's DPX1
                mask = np.random.rand(
                    CHROMOSOME_LENGTH
                ) < 0.5

                offspring = np.where(
                    mask, parent1, parent2
                )

                # Integer mutation: probability = 1/L
                for gene in range(CHROMOSOME_LENGTH):

                    if random.random() < (
                        1 / CHROMOSOME_LENGTH
                    ):

                        offspring[gene] = random.randint(
                            MIN_GREEN,
                            MAX_GREEN
                        )

                # Evaluate offspring
                offspring_fitness = fitness_function(
                    offspring
                )

                # Replace if offspring is not worse
                if offspring_fitness <= old_fitness[x, y]:

                    new_population[x, y] = offspring
                    new_fitness[x, y] = offspring_fitness

        # Update the complete population simultaneously
        population = new_population
        fitness_grid = new_fitness

        # Find the best solution in this generation
        best_index = np.unravel_index(
            np.argmin(fitness_grid),
            fitness_grid.shape
        )

        best_fitness = fitness_grid[best_index]

        best_solution = population[
            best_index[0], best_index[1]
        ]

        # Display only 25% of generations
        # First 5, then every 5th generation
        if generation < 5 or (generation + 1) % 5 == 0:

            print(
                f"Generation {generation + 1}: "
                f"Best Fitness = {best_fitness:.2f}"
            )

    return best_solution, best_fitness


best_solution, best_fitness = cellular_GA()


print("\n========== FINAL BEST SOLUTION ==========")

print("Best Fitness:", round(best_fitness, 2))

print("\nOptimized Traffic Signal Timings:")

for i in range(INTERSECTIONS):

    ns_green = best_solution[i * 2]
    ew_green = best_solution[i * 2 + 1]

    print(f"\nIntersection {i + 1}:")

    print(
        f"North-South Green: {ns_green} seconds"
    )

    print(
        f"North-South Yellow: {YELLOW_TIME} seconds"
    )

    print(
        f"East-West Green: {ew_green} seconds"
    )

    print(
        f"East-West Yellow: {YELLOW_TIME} seconds"
    )

    print(
        f"Total Cycle Time: "
        f"{ns_green + ew_green + 2 * YELLOW_TIME} seconds"
    )


Generation 1: Best Fitness = 151018.00
Generation 2: Best Fitness = 150683.00
Generation 3: Best Fitness = 149518.00
Generation 4: Best Fitness = 148969.00
Generation 5: Best Fitness = 148518.00
Generation 10: Best Fitness = 147765.00
Generation 15: Best Fitness = 147432.00
Generation 20: Best Fitness = 147012.00
Generation 25: Best Fitness = 146468.00
Generation 30: Best Fitness = 146351.00
Generation 35: Best Fitness = 146351.00
Generation 40: Best Fitness = 146212.00
Generation 45: Best Fitness = 146148.00
Generation 50: Best Fitness = 146127.00
Generation 55: Best Fitness = 146015.00
Generation 60: Best Fitness = 145983.00
Generation 65: Best Fitness = 145968.00
Generation 70: Best Fitness = 145968.00
Generation 75: Best Fitness = 145967.00
Generation 80: Best Fitness = 145967.00
Generation 85: Best Fitness = 145967.00
Generation 90: Best Fitness = 145967.00
Generation 95: Best Fitness = 145967.00
Generation 100: Best Fitness = 145967.00

========== FINAL BEST SOLUTION ==========
B